In [11]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset, Audio

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    batch_size=4,
    torch_dtype=torch_dtype,
    device=device,
    generate_kwargs={
        "language": "ar",          # 🔥 FORCE ARABIC
        "task": "transcribe",
        "temperature": 0.0 # (not translate)
    }
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [10]:
from google.colab import drive
drive.mount('/content/drive')

from datasets import load_from_disk
# Please make sure you have added the shared Google Drive folder to your 'My Drive'
# If the folder name is 'final' in your 'My Drive', the path should be:
DATASET_PATH  = "/content/drive/My Drive/final"
dataset = load_from_disk(str(DATASET_PATH))
print(dataset)

print(f"\nTrain : {len(dataset['train']):,} segments")
print(f"Test  : {len(dataset['test']):,}  segments")
print(f"\nFeatures: {dataset['train'].features}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DatasetDict({
    train: Dataset({
        features: ['audio_id', 'audio', 'transcript', 'duration_s', 'seg_start', 'seg_end'],
        num_rows: 72314
    })
    test: Dataset({
        features: ['audio_id', 'audio', 'transcript', 'duration_s', 'seg_start', 'seg_end'],
        num_rows: 1005
    })
})

Train : 72,314 segments
Test  : 1,005  segments

Features: {'audio_id': Value('string'), 'audio': Audio(sampling_rate=16000, decode=False, stream_index=None), 'transcript': Value('string'), 'duration_s': Value('float32'), 'seg_start': Value('float32'), 'seg_end': Value('float32')}


In [13]:
test_dataset = dataset["test"].remove_columns([
    "audio_id",
    "seg_start",
    "seg_end"
])

print(test_dataset)

Dataset({
    features: ['audio', 'transcript', 'duration_s'],
    num_rows: 1005
})


In [16]:
from datasets import Audio

test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16000))

In [14]:
!pip install -q evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.0 MB/s eta 0:00:00


In [17]:
import evaluate
from tqdm.auto import tqdm
import numpy as np
import time
import re

# =========================
# Metrics
# =========================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# =========================
# Cleaning function
# =========================
chars_to_ignore_regex = r'[\,\?\.\!\-\;\:\"\‘\’\“\”]'

def remove_special_characters(batch):
    text = batch["transcript"]
    text = re.sub(chars_to_ignore_regex, '', text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    batch["transcript"] = text
    return batch

# =========================
# Prepare references
# =========================
references = test_dataset["transcript"]

# =========================
# Inference + timing
# =========================
predictions = []
latencies = []

print(f"Running inference on {len(test_dataset)} samples...")

start_total = time.time()

for x in tqdm(test_dataset, total=len(test_dataset)):

    input_audio = {
        "array": np.array(x["audio"]["array"]),
        "sampling_rate": 16000
    }

    start = time.time()
    result = pipe(input_audio)
    end = time.time()

    predictions.append(result["text"])
    latencies.append(end - start)

end_total = time.time()

# =========================
# Latency metrics
# =========================
total_time = end_total - start_total
avg_latency = total_time / len(predictions)
mean_latency = np.mean(latencies)
median_latency = np.median(latencies)

# =========================
# Optimized Total audio duration 🔥
# =========================
total_audio_duration = sum(test_dataset["duration_s"])

# =========================
# Real-Time Factor (RTF)
# =========================
rtf = total_time / total_audio_duration

# =========================
# Clean predictions & references
# =========================
predictions = [
    remove_special_characters({"transcript": p})["transcript"]
    for p in predictions
]

references = [
    remove_special_characters({"transcript": r})["transcript"]
    for r in references
]

# =========================
# Compute WER / CER
# =========================
final_wer = wer_metric.compute(predictions=predictions, references=references)
final_cer = cer_metric.compute(predictions=predictions, references=references)

# =========================
# Print results
# =========================
print("\n===== BENCHMARK RESULTS =====")
print(f"WER: {final_wer:.4f}")
print(f"CER: {final_cer:.4f}")
print(f"Total inference time: {total_time:.2f} sec")
print(f"Average latency per sample: {avg_latency:.4f} sec")
print(f"Mean latency: {mean_latency:.4f} sec")
print(f"Median latency: {median_latency:.4f} sec")
print(f"Total audio duration: {total_audio_duration:.2f} sec")
print(f"Real-Time Factor (RTF): {rtf:.4f}")

Running inference on 1005 samples...


  0%|          | 0/1005 [00:00<?, ?it/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
You seem to be using the pipeli


===== BENCHMARK RESULTS =====
WER: 0.9656
CER: 0.6114
Total inference time: 2452.54 sec
Average latency per sample: 2.4403 sec
Mean latency: 2.4315 sec
Median latency: 1.6173 sec
Total audio duration: 3956.92 sec
Real-Time Factor (RTF): 0.6198


In [18]:
import pandas as pd
results_df = pd.DataFrame({"Reference": references, "Prediction": predictions})
display(results_df.head(50))

,Reference,Prediction
0,ويرحمني و اياكم في الدنيا و الآخرة,ويرحمني وإياكم في الدنيا والآخرة
1,اشك اناهي الحكومة الي باش تدخل لك في اصلاحات خ...,أنا هي الحكومة اللي باش تدخلك في إصلاحات خطيرة
2,وهو قاعد ستة شهر,وهو قاعد ستة شهر
3,اذن نحن بحاجة الى مجلس تاسيسي ان ينجم ينظم كل ...,إذن نحن بحاجة إلى مجلس تأسيسي ينجي منظم كل هذا
4,الادارة العامة للميترو طالبة خدامة و الادارة ا...,مدار العامة للمتروط أربع خدمة ومدار العامة وال...
5,صندوق عجب صباح الخير بربي حبيت نتكلم بربي شوفو...,توقع جبس بأخير بربي حبيت نتكلم بربي شوفونا الأ...
6,كيفاش تعملت قوانينو آش بيها المنظومة هذيكة متا...,كيفاش تعملت قوانين وشباه المنظومة دي كانت على ...
7,اقعد معايا سليم خلي ناخذوا جمال معانا بالتليفو...,مع ياسيم خلني اخدو جميل معانا بالتليفون جميل؟
8,معانا محرز بالتليفون مرحبا,معنا محرز بالتليفون مرحبا
9,كيفاش ينجم يكون القرار هذايا ناجح ويوصل للاقلا...,كيفاش نجم يكون القرار هذي ناجح ويوصل للإقلاع ا...


In [19]:
results_df = pd.DataFrame({
    "Reference": references,
    "Prediction": predictions
})

results_df.to_csv("whisper_large_v3_turbo_results.csv", index=False)